In [ ]:
import importlib
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from openai.types.chat import ChatCompletionUserMessageParam
from openai.types.chat import ChatCompletionMessageParam


<module 'utils.scraper' from 'e:\\avl-sessions\\ai_engineering_core\\week1\\utils\\scraper.py'>

In [ ]:
import utils.scraper

importlib.reload(utils.scraper)


In [4]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

# check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [6]:
message = "Hi GPT, how are you today? this is my first ever call to the frontier model"
messages: list[ChatCompletionUserMessageParam] = [
    {
    "role": "user",
    "content": message
    }
]
messages

[{'role': 'user',
  'content': 'Hi GPT, how are you today? this is my first ever call to the frontier model'}]

# `response` contains the complete response object returned by the OpenAI API.
#
# response
# ├── id                  → Unique ID of the API response
# ├── model               → Model used for generation
# ├── choices             → List of generated responses
# │   └── [0]
# │       ├── message
# │       │   ├── role    → Usually "assistant"
# │       │   └── content → ⭐ Actual text generated by the LLM
# │       └── finish_reason → Why generation stopped
# │
# └── usage               → Token usage information
#     ├── prompt_tokens       → Tokens sent to the model
#     ├── completion_tokens   → Tokens generated by the model
#     └── total_tokens        → Total tokens used
#
# To get the actual LLM answer:

In [8]:
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages
)
print(response.choices[0].message.content)

Hi! I’m glad to meet you. I’m doing well, and I’m here to help with whatever you need on Frontier.

I can assist with a wide range of tasks, for example:
- Answer questions and explain concepts clearly
- Write, edit, and polish text (emails, essays, reports, etc.)
- Brainstorm ideas and plan projects
- Help with coding, debugging, and data tasks
- Create summaries, outlines, or analyses
- Translate or adapt content
- Step-by-step guides and tutorials

If you’d like, we can start with a quick demo. For instance:
- Explain a concept in simple terms
- Draft a short email or note
- Outline a learning plan for Python or another topic

Tell me your preferred style (concise, detailed, formal, casual) and what you’d like to work on today. What would you like to do first?


In [38]:
print(response.choices[0].finish_reason)

stop


In [10]:
usage = response.usage

if usage:
    print(usage.prompt_tokens)
    print(usage.completion_tokens)
    print(usage.total_tokens)

24
1024
1048


In [37]:
fetch_website_contents = utils.scraper.fetch_website_contents
ed = fetch_website_contents("https://edwarddonner.com");
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
F

In [45]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [44]:
system_prompt_helpful = """
You are a helpful assistant that analyzes the contents of a website
and provides a clear, concise, and useful summary.

Ignore navigation menus, advertisements, social media links, and
irrelevant boilerplate text.

If the website contains important news or announcements, summarize
them as well.

Do not make up information that is not present in the website content.
Respond in markdown. Do not wrap the markdown in a code block -
respond just with the markdown.
"""

In [40]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [43]:
messages: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": "You are a Helpful assistant."
    },
    {
        "role": "user",
        "content": "What is 2 + 2?"
    }
]

response = client.chat.completions.create(model="gpt-4.1-nano", messages=messages)
content = response.choices[0].message.content
display(Markdown(content))

2 + 2 equals 4.

In [55]:

def messages_for(website: str) -> list[ChatCompletionMessageParam]:
    return [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt_prefix + website,
        },
    ]

In [56]:
messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

In [57]:
def summarize(url):
    website = fetch_website_contents(url)
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [58]:

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [59]:
display_summary("https://edwarddonner.com")

# Edward Donner’s Website: Where AI Nerdiness Meets Modest Bragging

So, Ed is your friendly neighborhood AI junkie who codes, dabble in *very amateur* electronic music (don’t quit your day job, Ed), and pretends to understand Hacker News. He’s a startup warrior — co-founder and CTO of Nebula.io, plus he once sold his creation untapt (because who doesn’t like a good tech exit?).  

The guy can’t stop talking about Large Language Models, so he poured his endless monologues into some Udemy courses that somehow racked up 900,000 enrollments in nearly 200 countries. Guess people *do* want to hear him rant.

**Latest Drama/News:**  
- February 2026: Dropped AI Coder resources turning coders into "agentic engineers" (fancy words, still basically coding stuff).  
- January 2026: Launched AI Builder tutorials with n8n because agents and voice agents are apparently a thing now.  
- September 2025: MLOps track to deploy AI to production for the overachievers.  

If you want to see AI models duke it out in a “battle of diplomacy and deviousness,” there’s a fun little arena called *Outsmart*. Because what’s better than watching digital egos fight?

Contact Ed if you want a lecture or just to say hi — he’s got a digital avatar ready for a chat. 

**Summary:** A techie’s personal playground with lots of AI courses, startup flexes, and a secret hope the world finally cares about his electronic beats.

In [60]:
display_summary("https://cnn.com")

# CNN: Your One-Stop Shop for Global Angst and Ad Complaints

Welcome to CNN, the place where the world’s chaos is neatly categorized into endless tabs: US, World, Politics, Business, Health, Entertainment—you name it, they’ve got a disaster and a headline for it. Want to watch a video? Brace yourself for the ad speedbumps because apparently, the real breaking news is whether your ad freezes or if the video ever loads.

Oh, and they *really* want your feedback on those pesky ads—because what's more thrilling than rating how annoyed you are by a commercial?

Latest news? They’ve got the Ukraine-Russia and Israel-Hamas wars on repeat, because who needs new drama when you have ongoing global crises? Throw in some climate gloom, election hoops, and celebrity gossip, and you’ve got yourself a 24/7 news fest of doom, gloom, and distraction.

In short: CNN is like your chatty, slightly buggy friend who keeps you updated, whether you want it or not—with plenty of options for complaining about ads.

In [61]:
display_summary("https://anthropic.com")

# Anthropic: The AI Safety Nerds You Didn't Know You Needed

Anthropic is a public benefit corporation obsessed with making AI *safe* and *useful* instead of apocalyptic. They tackle the "hard questions" on AI safety, governance, economics, and societal impacts because apparently, robots taking over isn’t their idea of a good time. 

They’ve got a bunch of AI products and models with artsy names like Mythos, Fable, Opus, Sonnet, and Haiku—because why not? Latest headline: **Opus 5** dropped, boasting ninja-level coding skills and sharper agent smarts, presumably to outwit your average chatbot.

If you want AI that doesn’t immediately plot your digital demise but instead tries to play nicely, Anthropic wants you to give their Claude AI a whirl. Also, career nerds and event junkies are invited to dig into their Academy and tutorials.

In short: Anthropic is the AI safety squad making sure the robot uprising is just theoretical.

In [62]:
display_summary("https://openai.com")

# OpenAI: Nerding Out with AI Since Forever

This site is basically OpenAI’s brain dump where they flaunt their latest and greatest AI wizardry. Highlights include GPT-5.6, which apparently scales with your ambition (because who doesn’t want an AI that matches their ego?). They’re also dropping goodies like free access to GPT-5.6 Luna and bringing AI smarts to health and cyber defense.

**News Flash:**  
- Jalapeño model is fast and efficient—AI with a spicy kick!  
- They’ve got new Chief Revenue Officer Dali Rajic strutting the corporate runway.  
- Keeping pace with “cyber-critical” stuff because, duh, security is a thing.  
- Price-performance upgrades to GPT-5.6 because money shouldn’t slow down your AI dreams.

**Fun Stories:**  
- Training bots to cycle across Antarctica. Because regular cycling is so 2025.  
- Simulating black holes with Codex, proving AI is out here cracking the cosmos.  
- Racing sponsorships, because apparently AI loves a good adrenaline rush.

**Research Nerd-Outs:**  
- Disproved some fancy math conjecture (take that, humans!).  
- Launched life sciences research models, because AI now plays doctor and scientist.  

Basically, if AI was a soap opera, this site is the binge-worthy drama full of power moves, brainy breakthroughs, and just enough sci-fi flair to keep you intrigued.

In [63]:
display_summary("https://chethana-virajini-portfolio.vercel.app/")

# Chethana’s Coding Chronicles: The Portfolio Edition

Meet Chethana, a full-stack developer who’s mastered React, Next.js, and Node.js with a sprinkle of backend NestJS magic—armed with 2 years of experience and enough skills to build your dream app (or at least a really snazzy e-commerce site). When not wrangling code or optimizing UI/UX with Tailwind, she’s either binge-watching movies, reading, or bossing around her cats.

Projects include:
- A slick e-commerce app with user login (because no one likes random shoppers).
- An admin dashboard that tracks real-time sales—keeping bosses happy.
- A weather app that forecasts so you’re never caught without an umbrella.

Also dabbling in machine learning and system design because why not? No earth-shattering announcements, just good old-fashioned coding prowess wrapped in a neat React bow. Want to connect? Download the CV or say hi.